# MEN 4338 · Week 9 — Kriging maps and validation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W9_kriging_maps_validation.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Last week we kriged single points. A mine plan needs a **map**: an estimate on every node of a grid, and the
kriging variance that says where the map can be trusted. Because we know the true Bassam grades, we can also
check what textbooks claim: kriging is the best linear estimator, it smooths, and it has less
**conditional bias** than IDW.

**What you will practise:** kriging on a grid · variance maps · comparison with the truth · smoothing ·
cross-validation with standardised errors · conditional bias (slope of regression).

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('data/bassam_truth_grid.csv', 'bassam_truth_grid.csv'), ('tests/w9/q1.py', 'tests/q1.py'), ('tests/w9/q2.py', 'tests/q2.py'), ('tests/w9/q3.py', 'tests/q3.py'), ('tests/w9/q4.py', 'tests/q4.py'), ('tests/w9/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

def idw_estimate(x0, y0, x, y, v, power=2, nmax=12):
    """Inverse-distance estimate at (x0, y0) from the nmax closest samples."""
    d = np.hypot(x - x0, y - y0)
    k = np.argsort(d)[:nmax]
    if d[k[0]] < 1e-9:
        return float(v[k[0]])
    w = 1.0 / d[k] ** power
    return float((w * v[k]).sum() / w.sum())

def ok_point(x0, y0, x, y, v, gamma, nmax=16):
    """Ordinary kriging at (x0, y0) with the nmax nearest data.
    gamma(dx, dy) is the variogram model, a function of the separation vector.
    Returns (estimate, kriging variance)."""
    d = np.hypot(x - x0, y - y0)
    idx = np.argsort(d)[:nmax]
    xs, ys, vs = x[idx], y[idx], v[idx]
    n = len(idx)
    A = np.ones((n + 1, n + 1)); A[n, n] = 0
    A[:n, :n] = gamma(xs[:, None] - xs[None, :], ys[:, None] - ys[None, :])
    b = np.ones(n + 1); b[:n] = gamma(xs - x0, ys - y0)
    sol = np.linalg.solve(A, b)
    w, mu = sol[:n], sol[n]
    return float(w @ vs), float(w @ b[:n] + mu)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def gamma_bassam(dx, dy):
    return spherical(aniso_h(dx, dy, 45, 300, 150), 0.06, 0.20, 300)

samples = pd.read_csv("bassam_samples.csv")
x, y, au = samples["x"].to_numpy(), samples["y"].to_numpy(), samples["au_gpt"].to_numpy()
T = pd.read_csv("bassam_truth_grid.csv")["au_gpt"].to_numpy().reshape(100, 120)
print(ok_point(600, 500, x, y, au, gamma_bassam))

## 1. From points to maps

A map is just kriging repeated on every node. Each node has its own neighbourhood (here the 16 closest
samples), so the computation is a loop. 3000 nodes take a few seconds.

The kriging variance map shows the drilling pattern: low near holes, high in the gaps and at the edges.
It does **not** show where grades are high: it depends on geometry only.

## 2. Smoothing and conditional bias

Kriging estimates are **smoother** than reality: their variance is smaller than the true variance, because
each estimate is an average. For mine planning the important property is that, among the blocks
**estimated** at a given grade, the true grades are on average that same grade. Regress the true values on the
estimates: a slope of 1 means no conditional bias; a slope below 1 means high estimates are too high and low
estimates too low, which sends ore to the waste dump and waste to the mill.

## 3. Standardised errors

If the variogram is right, the kriging variance predicts the size of the errors. In cross-validation the
**mean standardised squared error** $\frac1n\sum e_i^2/\sigma^2_{OK,i}$ should be close to 1.

---
## Exercises

### Q1. The grid

Build the nodes `gx, gy` of a 20 m grid with `np.meshgrid(np.arange(15, 1200, 20), np.arange(15, 1000, 20))`
(they fall on the centres of the truth cells) and store their number in `n_nodes`.

In [ ]:
gx, gy = ...
n_nodes = ...
gx.shape, n_nodes

In [ ]:
grader.check("q1")

### Q2. Kriging every node

Fill `ok_map` and `ok_varmap` (same shape as `gx`) with `ok_point(..., gamma_bassam, nmax=16)`.

In [ ]:
ok_map = np.zeros(gx.shape)
ok_varmap = np.zeros(gx.shape)
...
ok_map.mean(), ok_varmap.min(), ok_varmap.max()

In [ ]:
grader.check("q2")

### Q3. Compare with the truth

`truth_nodes = T[1::2, 1::2]` are the true grades under the nodes. Store `mse_ok_truth`. Make the IDW map
`idw_map` (power 2, 12 samples) on the same nodes and store `mse_idw_truth`. Store `smoothing` =
variance of `ok_map` / variance of `truth_nodes`.

In [ ]:
truth_nodes = T[1::2, 1::2]
mse_ok_truth = ...
idw_map = ...
mse_idw_truth = ...
smoothing = ...
mse_ok_truth, mse_idw_truth, smoothing

In [ ]:
grader.check("q3")

### Q4. Cross-validation of kriging

Leave-one-out with OK (16 neighbours): store the errors in `ok_cv_err`, the MSE in `ok_cv_mse` and the mean
standardised squared error in `ok_cv_msse`. Compare `ok_cv_mse` with the IDW value of week 6.

In [ ]:
ok_cv_err = ...
ok_cv_mse = ...
ok_cv_msse = ...
ok_cv_mse, ok_cv_msse

In [ ]:
grader.check("q4")

### Q5. Conditional bias and maps

Store the slope of the regression of the true values on the estimates for kriging (`slope_ok`) and for IDW
(`slope_idw`) with `np.polyfit(estimates.ravel(), truth_nodes.ravel(), 1)[0]`. Then draw `truth_nodes`,
`ok_map` and `ok_varmap` side by side: `fig_maps, axes_maps = plt.subplots(1, 3, figsize=(15, 4))`.

In [ ]:
slope_ok = ...
slope_idw = ...

fig_maps, axes_maps = plt.subplots(1, 3, figsize=(15, 4))
...
plt.show()
slope_ok, slope_idw

In [ ]:
grader.check("q5")

### Going further (not checked)

* Krig with 4, 8, 16 and 32 neighbours. How do `mse_ok_truth` and `slope_ok` change? Small neighbourhoods
  increase conditional bias.
* Plot the absolute error |ok_map − truth| against `ok_varmap`. Does a high kriging variance mean a large error?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()